# ML: Forward Stockout Prediction

Predicts stockout risk with a distributed Spark `GBTClassifier` and a held-out
isotonic probability calibrator.

## Model contract
- Historical training snapshots are the last inventory state for each
  store/product/day.
- Labels use stockout events on future dates only, through the configured
  horizon.
- Train, probability-calibration, and test labels are partitioned by label
  availability with a three-day horizon embargo between partitions.
- `stockout_probability` is calibrated; the raw GBT score is never published
  as a probability.
- Gold output preserves each position's source timestamps, records publication
  time in `generated_at`, and contains one row per store/product/horizon.


In [ ]:
from datetime import timedelta
import os

import mlflow
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import IsotonicRegression
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
from pyspark.sql.window import Window


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================


def get_env(var_name, default=None):
    return os.environ.get(var_name, default)


LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="ag")
GOLD_DB = get_env("GOLD_DB", default="au")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="stockout_prediction")
INVENTORY_TXN_TABLE = get_env(
    "INVENTORY_TXN_TABLE", default="fact_store_inventory_txn"
)
RECEIPT_LINES_TABLE = get_env(
    "RECEIPT_LINES_TABLE", default="fact_receipt_lines"
)
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
PRODUCTS_TABLE = get_env("PRODUCTS_TABLE", default="dim_products")
STOCKOUT_RISK_TABLE = get_env("STOCKOUT_RISK_TABLE", default="stockout_risk")
STOCKOUT_RISK_TABLE_NAME = (
    f"{LAKEHOUSE_NAME}.{GOLD_DB}.{STOCKOUT_RISK_TABLE}"
)
SCHEDULE_CADENCE = get_env("SCHEDULE_CADENCE", default="daily")
LABEL_COL = "stockout_label"

FORECAST_HORIZON_DAYS = int(
    get_env("FORECAST_HORIZON_DAYS", default="3")
)
LOOKBACK_DAYS = int(get_env("LOOKBACK_DAYS", default="30"))
TRAINING_HISTORY_DAYS = int(
    get_env("TRAINING_HISTORY_DAYS", default="180")
)
STOCKOUT_THRESHOLD = float(get_env("STOCKOUT_THRESHOLD", default="0"))
TRAIN_FRACTION = float(get_env("TRAIN_FRACTION", default="0.6"))
CALIBRATION_FRACTION = float(
    get_env("CALIBRATION_FRACTION", default="0.2")
)

MAX_ITER = int(get_env("MAX_ITER", default="80"))
MAX_DEPTH = int(get_env("MAX_DEPTH", default="5"))
STEP_SIZE = float(get_env("STEP_SIZE", default="0.1"))
SUBSAMPLING_RATE = float(get_env("SUBSAMPLING_RATE", default="1.0"))
RANDOM_SEED = int(get_env("RANDOM_SEED", default="42"))

if min(FORECAST_HORIZON_DAYS, LOOKBACK_DAYS, TRAINING_HISTORY_DAYS) < 1:
    raise ValueError("Forecast, feature, and training windows must be positive.")
if not 0.0 < TRAIN_FRACTION < 1.0:
    raise ValueError("TRAIN_FRACTION must be between 0 and 1.")
if not 0.0 < CALIBRATION_FRACTION < 1.0:
    raise ValueError("CALIBRATION_FRACTION must be between 0 and 1.")
if TRAIN_FRACTION + CALIBRATION_FRACTION >= 1.0:
    raise ValueError("Train and calibration fractions must leave a test window.")

print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Output table: {STOCKOUT_RISK_TABLE_NAME}")
print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} days")
print(f"Demand feature lookback: {LOOKBACK_DAYS} days")
print(f"Training history: {TRAINING_HISTORY_DAYS} days")

# Physical ML contract used by repository validation and the pre-write gate.
ML_SOURCE_TABLES = ('fact_store_inventory_txn', 'fact_receipt_lines', 'fact_receipts', 'dim_products')
ML_OUTPUT_CONTRACTS = {'stockout_risk': [('store_id', 'long', False),
                   ('product_id', 'long', False),
                   ('inventory_as_of', 'timestamp', False),
                   ('current_inventory', 'long', False),
                   ('demand_velocity_daily', 'double', False),
                   ('days_of_inventory', 'double', False),
                   ('demand_trend', 'double', False),
                   ('stockout_probability', 'double', False),
                   ('stockout_predicted', 'int', False),
                   ('risk', 'string', False),
                   ('ranking', 'int', False),
                   ('risk_level', 'string', False),
                   ('predicted_at', 'timestamp', False),
                   ('generated_at', 'timestamp', False),
                   ('forecast_horizon_days', 'int', False),
                   ('Department', 'string', True),
                   ('Category', 'string', True),
                   ('Subcategory', 'string', True),
                   ('model_run_id', 'string', False),
                   ('schema_version', 'string', False)]}


In [ ]:
# =============================================================================
# HELPERS & MLFLOW SETUP
# =============================================================================


def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")


def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")


def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False


def resolve_table_column(frame, table_name, *candidates):
    available = {column.lower(): column for column in frame.columns}
    for candidate in candidates:
        resolved = available.get(candidate.lower())
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Unable to resolve any of {candidates} in {table_name}. "
        f"Available columns: {frame.columns}"
    )


def chronological_split_boundaries(
    ordered_dates,
    train_fraction=0.6,
    calibration_fraction=0.2,
    embargo_days=0,
):
    """Return chronological label-availability splits with horizon embargoes."""
    dates = sorted(set(ordered_dates))
    if len(dates) < 3:
        raise ValueError(
            "Chronological train/calibration/test requires at least 3 dates."
        )
    if embargo_days < 0:
        raise ValueError("Embargo days cannot be negative.")
    usable_span_days = (dates[-1] - dates[0]).days - (2 * embargo_days)
    if usable_span_days < 3:
        raise ValueError("Insufficient label-availability span after embargoes.")

    train_days = max(1, int(usable_span_days * train_fraction))
    calibration_days = max(1, int(usable_span_days * calibration_fraction))
    train_target = dates[0] + timedelta(days=train_days)
    train_end = max(date_value for date_value in dates if date_value <= train_target)
    calibration_start = next(
        (
            date_value
            for date_value in dates
            if date_value > train_end + timedelta(days=embargo_days)
        ),
        None,
    )
    if calibration_start is None:
        raise ValueError("No calibration dates remain after the training embargo.")
    calibration_target = calibration_start + timedelta(days=calibration_days)
    calibration_end = max(
        date_value
        for date_value in dates
        if calibration_start <= date_value <= calibration_target
    )
    test_start = next(
        (
            date_value
            for date_value in dates
            if date_value > calibration_end + timedelta(days=embargo_days)
        ),
        None,
    )
    if test_start is None:
        raise ValueError("No test dates remain after the calibration embargo.")
    return {
        "train_end": train_end,
        "calibration_start": calibration_start,
        "calibration_end": calibration_end,
        "test_start": test_start,
    }


def assert_unique_keys(frame, key_columns, context):
    duplicates = (
        frame.groupBy(*key_columns)
        .count()
        .filter(F.col("count") != 1)
        .limit(1)
        .count()
    )
    if duplicates:
        raise RuntimeError(f"{context} is not unique by {key_columns}.")


def require_binary_classes(frame, label_column, context):
    if frame.select(label_column).distinct().count() != 2:
        raise RuntimeError(
            f"{context} must contain both classes for fail-closed training."
        )


def build_end_of_day_inventory(inventory_events):
    """Select the final inventory state per store, product, and source day."""
    end_of_day_window = Window.partitionBy(
        "store_id", "product_id", "event_date"
    ).orderBy(
        F.desc("event_ts"),
        F.desc_nulls_last("trace_id"),
        F.desc("balance"),
    )
    return (
        inventory_events.withColumn(
            "end_of_day_row", F.row_number().over(end_of_day_window)
        )
        .filter(F.col("end_of_day_row") == 1)
        .select(
            "store_id",
            "product_id",
            F.col("event_date").alias("snapshot_date"),
            F.col("event_ts").alias("inventory_as_of"),
            F.col("balance").alias("current_inventory"),
        )
    )


def build_snapshot_features(snapshot_frame, sales, products):
    """Build trailing demand features as of each inventory timestamp."""
    snapshot_sales = (
        snapshot_frame.alias("snapshot")
        .join(
            sales.alias("sale"),
            on=(
                (F.col("snapshot.store_id") == F.col("sale.store_id"))
                & (
                    F.col("snapshot.product_id")
                    == F.col("sale.product_id")
                )
                & (
                    F.col("sale.sales_event_ts")
                    <= F.col("snapshot.inventory_as_of")
                )
                & (
                    F.col("sale.sales_event_date")
                    >= F.date_sub(
                        F.col("snapshot.snapshot_date"), LOOKBACK_DAYS - 1
                    )
                )
            ),
            how="left",
        )
        .select(
            F.col("snapshot.store_id").alias("store_id"),
            F.col("snapshot.product_id").alias("product_id"),
            F.col("snapshot.snapshot_date").alias("snapshot_date"),
            F.col("snapshot.inventory_as_of").alias("inventory_as_of"),
            F.col("snapshot.current_inventory").alias("current_inventory"),
            F.col("sale.quantity").alias("quantity"),
            F.datediff(
                F.col("snapshot.snapshot_date"),
                F.col("sale.sales_event_date"),
            ).alias("days_ago"),
        )
    )
    demand_by_snapshot = snapshot_sales.groupBy(
        "store_id",
        "product_id",
        "snapshot_date",
        "inventory_as_of",
        "current_inventory",
    ).agg(
        F.sum(
            F.when(F.col("days_ago") < 7, F.col("quantity")).otherwise(0)
        ).alias("demand_7d"),
        F.sum(
            F.when(F.col("days_ago") < 14, F.col("quantity")).otherwise(0)
        ).alias("demand_14d"),
        F.sum(
            F.when(F.col("days_ago") < 30, F.col("quantity")).otherwise(0)
        ).alias("demand_30d"),
        F.avg(
            F.when(F.col("days_ago") < 30, F.col("quantity"))
        ).alias("avg_order_size_30d"),
    )
    return (
        demand_by_snapshot.join(products, on="product_id", how="left")
        .fillna(
            0.0,
            subset=[
                "demand_7d",
                "demand_14d",
                "demand_30d",
                "avg_order_size_30d",
            ],
        )
        .withColumn("demand_velocity_daily", F.col("demand_30d") / 30.0)
        .withColumn("day_of_week", F.dayofweek("snapshot_date"))
        .withColumn("day_of_month", F.dayofmonth("snapshot_date"))
        .withColumn("week_of_year", F.weekofyear("snapshot_date"))
        .withColumn(
            "is_weekend",
            F.when(F.dayofweek("snapshot_date").isin(1, 7), 1).otherwise(0),
        )
        .withColumn(
            "days_of_inventory",
            F.when(
                F.col("demand_velocity_daily") > 0,
                F.col("current_inventory") / F.col("demand_velocity_daily"),
            ).otherwise(999.0),
        )
        .withColumn(
            "demand_trend",
            F.when(
                F.col("demand_30d") > 0,
                (F.col("demand_7d") / 7.0)
                / (F.col("demand_30d") / 30.0),
            ).otherwise(1.0),
        )
        .withColumn(
            "inventory_ratio",
            F.when(
                F.col("demand_30d") > 0,
                F.col("current_inventory") / F.col("demand_30d"),
            ).otherwise(1.0),
        )
    )


def attach_future_stockout_labels(snapshot_features, stockout_events):
    """Label stockouts on dates strictly after each end-of-day snapshot."""
    future_hits = (
        snapshot_features.select(
            "store_id", "product_id", "snapshot_date", "inventory_as_of"
        )
        .alias("snapshot")
        .join(
            stockout_events.alias("stockout"),
            on=(
                (F.col("snapshot.store_id") == F.col("stockout.store_id"))
                & (
                    F.col("snapshot.product_id")
                    == F.col("stockout.product_id")
                )
                & (
                    F.col("stockout.stockout_date")
                    > F.col("snapshot.snapshot_date")
                )
                & (
                    F.col("stockout.stockout_date")
                    <= F.date_add(
                        F.col("snapshot.snapshot_date"),
                        FORECAST_HORIZON_DAYS,
                    )
                )
            ),
            how="left",
        )
        .select(
            F.col("snapshot.store_id").alias("store_id"),
            F.col("snapshot.product_id").alias("product_id"),
            F.col("snapshot.snapshot_date").alias("snapshot_date"),
            F.col("snapshot.inventory_as_of").alias("inventory_as_of"),
            F.col("stockout.stockout_date").alias("stockout_date"),
        )
        .groupBy(
            "store_id", "product_id", "snapshot_date", "inventory_as_of"
        )
        .agg(F.count("stockout_date").alias("future_stockout_count"))
        .withColumn(
            LABEL_COL,
            (F.col("future_stockout_count") > 0).cast("double"),
        )
        .drop("future_stockout_count")
    )
    return snapshot_features.join(
        future_hits,
        on=["store_id", "product_id", "snapshot_date", "inventory_as_of"],
        how="inner",
    ).withColumn(
        "label_available_date",
        F.date_add(F.col("snapshot_date"), FORECAST_HORIZON_DAYS),
    )


def raw_stockout_scores(model, frame):
    return model.transform(frame).withColumn(
        "raw_stockout_score",
        vector_to_array("probability")[1],
    )


def score_with_calibration(model, calibrator, calibration_assembler, frame):
    raw_scores = raw_stockout_scores(model, frame)
    calibrated = calibrator.transform(
        calibration_assembler.transform(raw_scores)
    ).withColumn(
        "stockout_probability",
        F.greatest(
            F.lit(0.0),
            F.least(F.lit(1.0), F.col("calibrated_probability")),
        ),
    )
    invalid_count = calibrated.filter(
        F.col("stockout_probability").isNull()
        | (F.col("stockout_probability") < 0.0)
        | (F.col("stockout_probability") > 1.0)
    ).limit(1).count()
    if invalid_count:
        raise RuntimeError("Calibrated stockout probabilities are invalid.")
    return calibrated.withColumn(
        "stockout_predicted",
        (F.col("stockout_probability") >= 0.5).cast("double"),
    )


ensure_database(GOLD_DB)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment '{EXPERIMENT_NAME}' ready")

def _normalize_ml_type(data_type):
    value = data_type.replace("bigint", "long").replace("integer", "int")
    return value


def validate_ml_output(frame, table_name):
    contract = ML_OUTPUT_CONTRACTS[table_name]
    expected = tuple((name, data_type) for name, data_type, _ in contract)
    actual = tuple(
        (field.name, _normalize_ml_type(field.dataType.simpleString()))
        for field in frame.schema.fields
    )
    if actual != expected:
        raise RuntimeError(
            f"ML output schema mismatch for {table_name}: expected={expected}, actual={actual}"
        )
    non_nullable = tuple(name for name, _, nullable in contract if not nullable)
    null_flags = frame.agg(
        *(F.max(F.col(name).isNull().cast("int")).alias(name) for name in non_nullable)
    ).first().asDict()
    null_columns = [name for name in non_nullable if null_flags.get(name)]
    if null_columns:
        raise RuntimeError(
            f"ML output {table_name} has nulls in required columns: {null_columns}"
        )
    return frame



## Step 1: Calculate Current Inventory Position and Historical Demand


In [ ]:
required_tables = [
    INVENTORY_TXN_TABLE,
    RECEIPT_LINES_TABLE,
    RECEIPTS_TABLE,
    PRODUCTS_TABLE,
]
for table_name in required_tables:
    if not silver_exists(table_name):
        raise RuntimeError(f"{table_name} table not found in Silver layer")

print("=" * 80)
print("EXTRACTING END-OF-DAY INVENTORY AND DEMAND DATA")
print("=" * 80)

inventory_history = (
    read_silver(INVENTORY_TXN_TABLE)
    .select(
        "store_id",
        "product_id",
        F.col("event_ts").cast("timestamp").alias("event_ts"),
        F.col("trace_id").cast("string").alias("trace_id"),
        F.col("balance").cast("double").alias("balance"),
    )
    .filter(
        F.col("store_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("event_ts").isNotNull()
    )
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn(
        "is_stockout",
        (F.col("balance") <= STOCKOUT_THRESHOLD).cast("int"),
    )
    .cache()
)
source_anchor = inventory_history.agg(
    F.max("event_ts").alias("source_as_of_timestamp")
).first()["source_as_of_timestamp"]
if source_anchor is None:
    raise RuntimeError(f"No inventory history found in {INVENTORY_TXN_TABLE}.")
source_as_of_timestamp = source_anchor
source_as_of_date = source_as_of_timestamp.date()
training_start_date = source_as_of_date - timedelta(
    days=TRAINING_HISTORY_DAYS
)
label_cutoff_date = source_as_of_date - timedelta(
    days=FORECAST_HORIZON_DAYS
)
sales_history_start_date = training_start_date - timedelta(days=LOOKBACK_DAYS)

inventory_eod = build_end_of_day_inventory(inventory_history).cache()
assert_unique_keys(
    inventory_eod,
    ["store_id", "product_id", "snapshot_date"],
    "End-of-day inventory",
)

latest_position_window = Window.partitionBy(
    "store_id", "product_id"
).orderBy(F.desc("snapshot_date"), F.desc("inventory_as_of"))
current_inventory_snapshots = (
    inventory_eod.withColumn(
        "current_row", F.row_number().over(latest_position_window)
    )
    .filter(F.col("current_row") == 1)
    .drop("current_row")
)
historical_snapshots = inventory_eod.filter(
    (F.col("snapshot_date") >= F.lit(training_start_date))
    & (F.col("snapshot_date") <= F.lit(label_cutoff_date))
)
stockout_events = (
    inventory_history.filter(F.col("is_stockout") == 1)
    .select(
        "store_id",
        "product_id",
        F.col("event_date").alias("stockout_date"),
    )
    .distinct()
)

sales_df = (
    read_silver(RECEIPT_LINES_TABLE)
    .select(
        "receipt_id_ext",
        "product_id",
        F.col("quantity").cast("double").alias("quantity"),
    )
    .alias("line")
    .join(
        read_silver(RECEIPTS_TABLE)
        .select(
            "receipt_id_ext",
            "store_id",
            F.col("event_ts").cast("timestamp").alias("sales_event_ts"),
        )
        .alias("receipt"),
        on="receipt_id_ext",
        how="inner",
    )
    .select(
        F.col("receipt.store_id").alias("store_id"),
        F.col("line.product_id").alias("product_id"),
        "quantity",
        "sales_event_ts",
        F.to_date("sales_event_ts").alias("sales_event_date"),
    )
    .filter(
        (F.col("sales_event_date") >= F.lit(sales_history_start_date))
        & (F.col("sales_event_date") <= F.lit(source_as_of_date))
    )
    .cache()
)

print(f"Source inventory as-of: {source_as_of_timestamp}")
print(f"End-of-day inventory rows: {inventory_eod.count()}")
print(f"Historical training snapshots: {historical_snapshots.count()}")
print(f"Current inventory positions: {current_inventory_snapshots.count()}")


## Step 2: Feature Engineering


In [ ]:
print("=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)

products_source = read_silver(PRODUCTS_TABLE)
product_id_column = resolve_table_column(
    products_source, PRODUCTS_TABLE, "product_id", "id"
)
department_column = resolve_table_column(
    products_source, PRODUCTS_TABLE, "department"
)
category_column = resolve_table_column(
    products_source, PRODUCTS_TABLE, "product_category", "category"
)
subcategory_column = resolve_table_column(
    products_source,
    PRODUCTS_TABLE,
    "product_subcategory",
    "subcategory",
)
products_df = (
    products_source.select(
        F.col(product_id_column).cast("long").alias("product_id"),
        F.col(department_column).alias("Department"),
        F.col(category_column).alias("Category"),
        F.col(subcategory_column).alias("Subcategory"),
    )
    .groupBy("product_id")
    .agg(
        F.min("Department").alias("Department"),
        F.min("Category").alias("Category"),
        F.min("Subcategory").alias("Subcategory"),
    )
)

features_df = build_snapshot_features(
    current_inventory_snapshots,
    sales_df,
    products_df,
).cache()
historical_features_df = build_snapshot_features(
    historical_snapshots,
    sales_df,
    products_df,
).cache()

assert_unique_keys(
    features_df,
    ["store_id", "product_id"],
    "Current stockout features",
)
assert_unique_keys(
    historical_features_df,
    ["store_id", "product_id", "snapshot_date"],
    "Historical stockout features",
)

print(f"Current feature rows: {features_df.count()}")
print(f"Historical end-of-day feature rows: {historical_features_df.count()}")
features_df.select(
    "store_id",
    "product_id",
    "inventory_as_of",
    "current_inventory",
    "demand_velocity_daily",
    "days_of_inventory",
    "demand_trend",
).show(10)


## Step 3: Create Training Labels (Historical Stockouts)


In [ ]:
print("=" * 80)
print("CREATING FUTURE-ONLY TRAINING LABELS")
print("=" * 80)

training_data = attach_future_stockout_labels(
    historical_features_df,
    stockout_events,
).cache()
assert_unique_keys(
    training_data,
    ["store_id", "product_id", "snapshot_date"],
    "Labeled stockout snapshots",
)

ordered_label_available_dates = [
    row["label_available_date"]
    for row in training_data.select("label_available_date")
    .distinct()
    .orderBy("label_available_date")
    .collect()
]
split_dates = chronological_split_boundaries(
    ordered_label_available_dates,
    TRAIN_FRACTION,
    CALIBRATION_FRACTION,
    FORECAST_HORIZON_DAYS,
)
train_label_available_end = split_dates["train_end"]
calibration_label_available_start = split_dates["calibration_start"]
calibration_label_available_end = split_dates["calibration_end"]
test_label_available_start = split_dates["test_start"]

print(f"Training records at end-of-day grain: {training_data.count()}")
training_data.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show()
print(f"Train labels available through: {train_label_available_end}")
print(f"Calibration labels start after embargo: {calibration_label_available_start}")
print(f"Calibration labels available through: {calibration_label_available_end}")
print(f"Test labels available from: {test_label_available_start}")


## Step 4: Train GBTClassifier Model (Distributed)

Uses Spark ML `GBTClassifier` — training runs distributed across Spark
executors.  All data stays in Spark DataFrames (no `toPandas()`).


In [ ]:
print("=" * 80)
print("TRAINING AND CALIBRATING GBTCLASSIFIER")
print("=" * 80)

feature_cols = [
    "current_inventory",
    "demand_7d",
    "demand_14d",
    "demand_30d",
    "demand_velocity_daily",
    "avg_order_size_30d",
    "days_of_inventory",
    "demand_trend",
    "inventory_ratio",
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "is_weekend",
]


def prepare_model_frame(frame):
    prepared = frame
    for column_name in feature_cols:
        prepared = prepared.withColumn(
            column_name,
            F.col(column_name).cast("double"),
        )
    return prepared.na.fill(0.0, subset=feature_cols)


assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="error",
)
assembled_history = assembler.transform(prepare_model_frame(training_data))
df_train = assembled_history.filter(
    F.col("label_available_date") <= F.lit(train_label_available_end)
)
df_calibration = assembled_history.filter(
    (F.col("label_available_date") >= F.lit(calibration_label_available_start))
    & (
        F.col("label_available_date")
        <= F.lit(calibration_label_available_end)
    )
)
df_test = assembled_history.filter(
    F.col("label_available_date") >= F.lit(test_label_available_start)
)

for split_name, split_frame in [
    ("Training split", df_train),
    ("Calibration split", df_calibration),
    ("Test split", df_test),
]:
    if split_frame.count() == 0:
        raise RuntimeError(f"{split_name} is empty.")
    require_binary_classes(split_frame, LABEL_COL, split_name)

train_count = df_train.count()
calibration_count = df_calibration.count()
test_count = df_test.count()
print(
    f"Chronological rows -- train: {train_count}, "
    f"calibration: {calibration_count}, test: {test_count}"
)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol=LABEL_COL,
    maxIter=MAX_ITER,
    maxDepth=MAX_DEPTH,
    stepSize=STEP_SIZE,
    subsamplingRate=SUBSAMPLING_RATE,
    seed=RANDOM_SEED,
)

with mlflow.start_run(
    run_name=f"gbt_stockout_{source_as_of_timestamp.strftime('%Y%m%d_%H%M')}"
) as run:
    model = gbt.fit(df_train.select("features", LABEL_COL))

    raw_calibration = raw_stockout_scores(model, df_calibration).cache()
    if raw_calibration.select("raw_stockout_score").distinct().count() < 2:
        raise RuntimeError(
            "Stockout calibration has fewer than two distinct model scores."
        )
    calibration_assembler = VectorAssembler(
        inputCols=["raw_stockout_score"],
        outputCol="calibration_features",
        handleInvalid="error",
    )
    calibrator = IsotonicRegression(
        featuresCol="calibration_features",
        labelCol=LABEL_COL,
        predictionCol="calibrated_probability",
        isotonic=True,
    ).fit(calibration_assembler.transform(raw_calibration))

    df_predictions = score_with_calibration(
        model,
        calibrator,
        calibration_assembler,
        df_test,
    )
    binary_evaluator = BinaryClassificationEvaluator(
        rawPredictionCol="stockout_probability",
        labelCol=LABEL_COL,
        metricName="areaUnderROC",
    )
    auc_roc = float(binary_evaluator.evaluate(df_predictions))
    multiclass_evaluator = MulticlassClassificationEvaluator(
        predictionCol="stockout_predicted",
        labelCol=LABEL_COL,
    )
    precision = float(
        multiclass_evaluator.evaluate(
            df_predictions,
            {multiclass_evaluator.metricName: "weightedPrecision"},
        )
    )
    recall = float(
        multiclass_evaluator.evaluate(
            df_predictions,
            {multiclass_evaluator.metricName: "weightedRecall"},
        )
    )
    f1 = float(
        multiclass_evaluator.evaluate(
            df_predictions,
            {multiclass_evaluator.metricName: "f1"},
        )
    )
    brier_score = float(
        df_predictions.agg(
            F.avg(
                F.pow(
                    F.col("stockout_probability") - F.col(LABEL_COL),
                    2,
                )
            ).alias("brier_score")
        ).first()["brier_score"]
    )

    mlflow.log_params({
        "algorithm": "gbt_classifier_isotonic_calibration",
        "forecast_horizon_days": FORECAST_HORIZON_DAYS,
        "lookback_days": LOOKBACK_DAYS,
        "training_history_days": TRAINING_HISTORY_DAYS,
        "max_iter": MAX_ITER,
        "max_depth": MAX_DEPTH,
        "step_size": STEP_SIZE,
        "subsampling_rate": SUBSAMPLING_RATE,
        "feature_count": len(feature_cols),
        "train_rows": train_count,
        "calibration_rows": calibration_count,
        "test_rows": test_count,
        "source_as_of": source_as_of_timestamp.isoformat(),
    })
    mlflow.log_metrics({
        "calibrated_auc_roc": round(auc_roc, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "brier_score": round(brier_score, 6),
    })
    mlflow.spark.log_model(model, "gbt_stockout_model")
    mlflow.spark.log_model(calibrator, "isotonic_stockout_calibrator")

    print(f"MLflow run: {run.info.run_id}")
    print(f"Calibrated test AUC-ROC: {auc_roc:.4f}")
    print(f"Precision: {precision:.4f}; Recall: {recall:.4f}; F1: {f1:.4f}")
    print(f"Brier score: {brier_score:.6f}")


## Step 5: Generate Predictions for Current Inventory

Score the full current-inventory feature set using the trained model.
All prediction stays in Spark — no `toPandas()`.


In [ ]:
print("=" * 80)
print("GENERATING CURRENT CALIBRATED STOCKOUT RISK")
print("=" * 80)

current_assembled = assembler.transform(prepare_model_frame(features_df))
df_scored = score_with_calibration(
    model,
    calibrator,
    calibration_assembler,
    current_assembled,
)

df_scored = (
    df_scored.withColumn(
        "risk_level",
        F.when(F.col("stockout_probability") >= 0.7, "HIGH")
        .when(F.col("stockout_probability") >= 0.4, "MEDIUM")
        .otherwise("LOW"),
    )
    .withColumn("risk", F.col("risk_level"))
    .withColumn(
        "predicted_at",
        F.lit(source_as_of_timestamp).cast("timestamp"),
    )
    .withColumn(
        "forecast_horizon_days",
        F.lit(FORECAST_HORIZON_DAYS).cast("int"),
    )
)

risk_rank_window = Window.orderBy(
    F.desc("stockout_probability"),
    F.asc("store_id"),
    F.asc("product_id"),
)
df_scored = df_scored.withColumn(
    "ranking", F.row_number().over(risk_rank_window)
)
assert_unique_keys(
    df_scored,
    ["store_id", "product_id", "forecast_horizon_days"],
    "Current stockout scores",
)

print(f"Predictions generated: {df_scored.count()}")
df_scored.groupBy("risk_level").count().orderBy("risk_level").show()
df_scored.select(
    "store_id",
    "product_id",
    "inventory_as_of",
    "current_inventory",
    "demand_velocity_daily",
    "days_of_inventory",
    "stockout_probability",
    "risk_level",
).orderBy(F.desc("stockout_probability")).show(20)


## Step 6: Save to Gold Layer


In [ ]:
print("=" * 80)
print("SAVING ONE CURRENT ROW PER STORE/PRODUCT/HORIZON")
print("=" * 80)

df_output = (
    df_scored.select(
        F.col("store_id").cast("long"),
        F.col("product_id").cast("long"),
        F.col("inventory_as_of").cast("timestamp"),
        F.col("current_inventory").cast("long"),
        F.col("demand_velocity_daily").cast("double"),
        F.col("days_of_inventory").cast("double"),
        F.col("demand_trend").cast("double"),
        F.col("stockout_probability").cast("double"),
        F.col("stockout_predicted").cast("int"),
        F.col("risk").cast("string"),
        F.col("ranking").cast("int"),
        F.col("risk_level").cast("string"),
        F.col("predicted_at").cast("timestamp"),
        F.current_timestamp().alias("generated_at"),
        F.col("forecast_horizon_days").cast("int"),
        F.col("Department").cast("string"),
        F.col("Category").cast("string"),
        F.col("Subcategory").cast("string"),
        F.lit(run.info.run_id).cast("string").alias("model_run_id"),
        F.lit("1.0").cast("string").alias("schema_version"),
    )
    .orderBy("store_id", "product_id", "forecast_horizon_days")
)
assert_unique_keys(
    df_output,
    ["store_id", "product_id", "forecast_horizon_days"],
    "Stockout Gold output",
)

df_output = validate_ml_output(df_output, "stockout_risk")
df_output.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(
    STOCKOUT_RISK_TABLE_NAME
)
row_count = spark.table(STOCKOUT_RISK_TABLE_NAME).count()
if row_count != current_inventory_snapshots.count():
    raise RuntimeError(
        "Published stockout output does not contain exactly one row per "
        "current store/product/horizon."
    )
print(f"{STOCKOUT_RISK_TABLE_NAME}: {row_count} rows")


In [ ]:
print("=" * 80)
print("STOCKOUT PREDICTION COMPLETE")
print("=" * 80)
print(f"Output table: {STOCKOUT_RISK_TABLE_NAME}")
print(f"Source as-of: {source_as_of_timestamp}")
print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} days")
print("Training grain: end-of-day store/product")
print("Published probabilities: isotonic calibrated")
print("Published inventory timestamp: inventory_as_of")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(
    f"Schedule this notebook on a {SCHEDULE_CADENCE} cadence for fresh "
    "predictions."
)
